# Position repair — retrain the corrupt seed, re-probe (Kaggle)

The Phase-3 sweep crashed on `position_strong` because the **seed-5 encoder
checkpoint is corrupt** (`backbone.pt` unreadable). Color and control already
have valid `stacks.npz`; only **position** needs redoing. This notebook, run top
to bottom, repairs just the position cell:

1. restore your uploaded position encoders,
2. **integrity-check them and purge any corrupt file** (so it gets retrained),
3. retrain only the missing/purged seeds,
4. re-probe position -> `results/probes/position_strong/stacks.npz`.

## Setup
1. **Settings -> Accelerator -> GPU T4** (a single T4 is enough; not P100 -- sm_60
   is unsupported by Kaggle's PyTorch).
2. **Settings -> Internet -> On.**
3. **Add Input -> your encoder dataset** (the `backbone_position_strong_seed*.pt`
   files you uploaded). It's fine if it still contains the corrupt seed 5 -- the
   integrity cell (section 5) purges and retrains it automatically.
4. Run sections 1-8 in order. Retraining one seed + re-probing is ~3-4 h on one T4,
   well within a session. It's idempotent: re-run after a session ends and it
   continues where it stopped.


## 1. Verify the GPU (expect one Tesla T4)

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Download dSprites + build the image cache
Position is the dSprites arm. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.dsprites --download --build-cache

## 5. Restore position encoders from your uploaded dataset
Add Input -> your dataset first, then run this. It finds every
`*position_strong_seed<N>*.pt` under `/kaggle/input` and restores it to
`results/encoders/position_strong_seed<N>/` (`backbone*.pt` -> `backbone.pt`,
`last_ckpt*.pt` -> `last_ckpt.pt`). Only position files are touched.

In [ ]:
import re, shutil
from pathlib import Path

REPO  = Path("/kaggle/working/probe-capacity-invariance")
ENC   = REPO / "results" / "encoders"
INPUT = Path("/kaggle/input")
_RID  = re.compile(r"position_strong_seed\d+")

def _target(name):
    n = name.lower()
    if "ckpt" in n:     return "last_ckpt.pt"
    if "backbone" in n: return "backbone.pt"
    return None

found = {}
for p in sorted(INPUT.rglob("*.pt")) if INPUT.exists() else []:
    m, tgt = _RID.search(p.as_posix()), _target(p.name)
    if m and tgt:
        found.setdefault((m.group(0), tgt), p)     # first match per (seed, kind)

if not found:
    print("No position_strong_seed*.pt under /kaggle/input -- Add Input -> your encoder dataset first.")
for (rid, tgt), src in sorted(found.items()):
    dst = ENC / rid / tgt
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dst)
    print(f"restored {rid}/{tgt:12s} <- {src.name}")

## 6. Integrity check -- purge corrupt encoders
Each restored `backbone.pt` is opened as a zip archive; any that is corrupt or
truncated is **deleted** (with its `last_ckpt.pt`) so section 7 retrains it from
scratch. This is what recovers seed 5, and guards against any other bad upload.

In [ ]:
import zipfile
from pathlib import Path

ENC = Path("/kaggle/working/probe-capacity-invariance/results/encoders")
purged, ok, missing = [], [], []
for d in sorted(ENC.glob("position_strong_seed*")):
    bb = d / "backbone.pt"
    if not bb.exists():
        missing.append(d.name); print(f"{d.name:26s} no backbone -> will train"); continue
    try:
        bad = zipfile.ZipFile(bb).testzip()
        reason = None if bad is None else f"corrupt ({bad})"
    except Exception as e:
        reason = f"not-a-zip ({e})"
    if reason is None:
        ok.append(d.name); print(f"{d.name:26s} OK")
    else:
        bb.unlink(missing_ok=True)
        (d / "last_ckpt.pt").unlink(missing_ok=True)
        purged.append(d.name); print(f"{d.name:26s} {reason} -> PURGED (will retrain)")
print(f"\nOK={len(ok)}  purged={len(purged)}  missing={len(missing)}  -> to train: {sorted(purged+missing)}")

## 7. Retrain the missing / purged position seeds
Single GPU, idempotent: a seed whose `backbone.pt` exists is skipped, so only the
purged/missing seeds train. Per-seed logs in `results/sweep_logs/`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
ENC  = REPO / "results" / "encoders"
LOGS = REPO / "results" / "sweep_logs"; LOGS.mkdir(parents=True, exist_ok=True)
CFG, SEEDS = "configs/run/position_strong.yaml", list(range(12))

def done(s): return (ENC / f"position_strong_seed{s}" / "backbone.pt").exists()

todo = [s for s in SEEDS if not done(s)]
print(f"position: {12 - len(todo)}/12 present | training {todo or 'nothing'}")
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
for s in todo:
    print(f"[train] position_strong_seed{s} ...", flush=True)
    with open(LOGS / f"position_strong_seed{s}.log", "a") as log:
        rc = subprocess.run(
            [sys.executable, "-m", "src.encoders.train_simclr", "--config", CFG,
             "--set", f"run.seed={s}", "run.num_workers=2", "run.device=cuda"],
            cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT).returncode
    print(f"[train] seed{s} exit {rc}" + ("" if rc == 0 else "  <-- FAILED (see log)"), flush=True)
print("present now:", sum(done(s) for s in SEEDS), "/12")

## 8. Re-probe position -> `stacks.npz`
Fits the capacity ladder + G/S/epsilon_G stacks across all 12 encoders and 12
random seeds, writing `results/probes/position_strong/{stacks.npz, meta.json}`.
~2.5 h on one T4. Needs all 12 encoders present (section 7).

In [ ]:
import glob, subprocess, time
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
encs = sorted(glob.glob(str(REPO / "results/encoders/position_strong_seed*/backbone.pt")))
print(f"{len(encs)}/12 position encoders found")
if len(encs) < 12:
    print("STOP: <12 encoders -- finish section 7 before probing."); raise SystemExit

stale = REPO / "results/probes/position_strong/stacks.npz"     # position never completed; recompute clean
if stale.exists():
    stale.unlink(); print("removed stale position stacks.npz")

cmd = ["python", "-m", "src.probes.run_sweep", "--config", "configs/probe/ladder.yaml",
       "--dataset", "dsprites", "--condition", "position", "--strength", "strong",
       "--encoders", *encs, "--random-seed", *[str(i) for i in range(12)],
       "--epochs", "100", "--num-workers", "2", "--data-path", "data/raw/dsprites.npz"]
t = time.time()
rc = subprocess.run(cmd, cwd=str(REPO)).returncode
print(f"\nprobe exit {rc} in {(time.time() - t) / 3600:.2f} h")

## 9. Package the output for download (collision-proof names)
Copies the position outputs to condition-prefixed names so they don't clash with
color/control `stacks.npz` in your browser. Download these two, then drop them in
`results/probes/position_strong/` locally.

In [ ]:
import shutil
from pathlib import Path

src = Path("/kaggle/working/probe-capacity-invariance/results/probes/position_strong")
for f, dst in [("stacks.npz", "position_stacks.npz"), ("meta.json", "position_meta.json")]:
    p = src / f
    print(f"download: {dst}" if p.exists() and shutil.copy(p, dst) else f"MISSING {p}")